In [1]:
# Libraries used for data preparation, prediction-error metrics, and classification evaluation
from sklearn import datasets
from sklearn.metrics import mean_absolute_error as MAE
from sklearn.metrics import mean_squared_error as MSE
from sklearn.metrics import mean_absolute_percentage_error as MAPE
from sklearn.metrics import classification_report as cr

import pandas as pd
import numpy as np
from numpy import percentile

## Data and Analytical Setup

The Iris dataset includes 150 observations and four continuous flower measurements: sepal length, sepal width, petal length, and petal width. Species is represented as a categorical outcome. The dataset was reviewed for structure, variable type, and missingness before analysis.

A derived feature was calculated as the product of sepal length and sepal width divided by the product of petal length and petal width. The data were then partitioned into training and test sets using an 80/20 stratified split to preserve species representation across both samples.

In [2]:
# Convert the sklearn Iris measurements to a DataFrame for analysis
iris = pd.DataFrame(datasets.load_iris().data)
iris

,0,1,2,3
0,5.1,3.5,1.4,0.2
1,4.9,3.0,1.4,0.2
2,4.7,3.2,1.3,0.2
3,4.6,3.1,1.5,0.2
4,5.0,3.6,1.4,0.2
...,...,...,...,...
145,6.7,3.0,5.2,2.3
146,6.3,2.5,5.0,1.9
147,6.5,3.0,5.2,2.0
148,6.2,3.4,5.4,2.3


In [3]:
iris.isna().sum()

0    0
1    0
2    0
3    0
dtype: int64

In [4]:
iris.dtypes

0    float64
1    float64
2    float64
3    float64
dtype: object

In [5]:
# Apply descriptive feature names and add the species target
iris.columns = datasets.load_iris().feature_names
iris['type'] = datasets.load_iris().target
iris.head()

,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm),type
0,5.1,3.5,1.4,0.2,0
1,4.9,3.0,1.4,0.2,0
2,4.7,3.2,1.3,0.2,0
3,4.6,3.1,1.5,0.2,0
4,5.0,3.6,1.4,0.2,0


In [6]:
# Treat species codes as categorical labels rather than continuous values
iris['type'] = iris['type'].astype('object')
iris.dtypes


sepal length (cm)    float64
sepal width (cm)     float64
petal length (cm)    float64
petal width (cm)     float64
type                  object
dtype: object

In [7]:
# Create the required ratio combining sepal and petal dimensions
iris['new'] = (iris.iloc[:,0] * iris.iloc[:,1])/(iris.iloc[:,2] * iris.iloc[:,3])
iris.head()

,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm),type,new
0,5.1,3.5,1.4,0.2,0,63.750000
1,4.9,3.0,1.4,0.2,0,52.500000
2,4.7,3.2,1.3,0.2,0,57.846154
3,4.6,3.1,1.5,0.2,0,47.533333
4,5.0,3.6,1.4,0.2,0,64.285714


In [8]:
from sklearn.model_selection import train_test_split as tts

# Preserve species proportions while reserving 20% of the data for evaluation
X_train, X_test, y_train, y_test = tts(
    iris.iloc[:, 0:4],
    iris.iloc[:, 4],
    test_size =  0.2,
    random_state = 42,
    stratify = iris.iloc[:, 4]
)

In [9]:
# Confirm class balance after stratification
y_train.value_counts()

type
0    40
2    40
1    40
Name: count, dtype: int64

In [10]:
# Confirm class balance in the held-out test set
y_test.value_counts()

type
0    10
2    10
1    10
Name: count, dtype: int64

In [11]:
# Verify that all four measurements are available for estimator construction
X_train.columns

Index(['sepal length (cm)', 'sepal width (cm)', 'petal length (cm)',
       'petal width (cm)'],
      dtype='object')

## Regression Analysis

Two constant estimators were evaluated for predicting sepal width. Estimator 1 used mean training petal length, whereas Estimator 2 used the mean difference between training sepal length and petal width. Performance was compared using mean error (ME), mean percentage error (MPE), mean absolute error (MAE), mean squared error (MSE), and mean absolute percentage error (MAPE).

Because both approaches generate a single constant prediction for all test observations, the comparison evaluates relative predictive performance rather than individualized modeling.

In [12]:
# Derive both constant estimators from training data only
est1 = np.mean(X_train['petal length (cm)'])
est2 = np.mean(X_train['sepal length (cm)'] - X_train['petal width (cm)'])

# Extend each constant estimate across the held-out observations
est1 = [est1] * len(y_test)
est2 = [est2] * len(y_test)

In [13]:
est1

[np.float64(3.77),
 np.float64(3.77),
 np.float64(3.77),
 np.float64(3.77),
 np.float64(3.77),
 np.float64(3.77),
 np.float64(3.77),
 np.float64(3.77),
 np.float64(3.77),
 np.float64(3.77),
 np.float64(3.77),
 np.float64(3.77),
 np.float64(3.77),
 np.float64(3.77),
 np.float64(3.77),
 np.float64(3.77),
 np.float64(3.77),
 np.float64(3.77),
 np.float64(3.77),
 np.float64(3.77),
 np.float64(3.77),
 np.float64(3.77),
 np.float64(3.77),
 np.float64(3.77),
 np.float64(3.77),
 np.float64(3.77),
 np.float64(3.77),
 np.float64(3.77),
 np.float64(3.77),
 np.float64(3.77)]

In [14]:
est2

[np.float64(4.636666666666667),
 np.float64(4.636666666666667),
 np.float64(4.636666666666667),
 np.float64(4.636666666666667),
 np.float64(4.636666666666667),
 np.float64(4.636666666666667),
 np.float64(4.636666666666667),
 np.float64(4.636666666666667),
 np.float64(4.636666666666667),
 np.float64(4.636666666666667),
 np.float64(4.636666666666667),
 np.float64(4.636666666666667),
 np.float64(4.636666666666667),
 np.float64(4.636666666666667),
 np.float64(4.636666666666667),
 np.float64(4.636666666666667),
 np.float64(4.636666666666667),
 np.float64(4.636666666666667),
 np.float64(4.636666666666667),
 np.float64(4.636666666666667),
 np.float64(4.636666666666667),
 np.float64(4.636666666666667),
 np.float64(4.636666666666667),
 np.float64(4.636666666666667),
 np.float64(4.636666666666667),
 np.float64(4.636666666666667),
 np.float64(4.636666666666667),
 np.float64(4.636666666666667),
 np.float64(4.636666666666667),
 np.float64(4.636666666666667)]

In [15]:
# Evaluate prediction bias and error magnitude using complementary regression metrics
def myf(y, yhat):
    ME = np.mean(y - yhat)
    MPE = np.mean((y -yhat) / y)
    myMAE = np.mean(MAE(y, yhat))
    myMSE = np.mean(MSE(y, yhat))
    myMAPE = np.mean(MAPE(y, yhat))

    print("ME:", np.round(ME, 3), 
          "MPE:",np.round(MPE, 3), 
          "myMAE:", np.round(myMAE, 3), 
          "myMSE:", np.round(myMSE, 3), 
          "myMAPE:", np.round(myMAPE, 3)
         )


In [16]:
# Compare both estimators against observed sepal width in the test set
myf(X_test['sepal width (cm)'], est1)
myf(X_test['sepal width (cm)'], est2)

ME: -0.677 MPE: -0.237 myMAE: 0.694 myMSE: 0.602 myMAPE: 0.242
ME: -1.543 MPE: -0.522 myMAE: 1.543 myMSE: 2.526 myMAPE: 0.522


### Regression Findings

Estimator 1 outperformed Estimator 2 across all five regression metrics. Its ME and MPE were closer to zero, indicating less directional bias, while its lower MAE, MSE, and MAPE indicated smaller prediction errors overall. Both estimators produced negative ME and MPE values, showing that they overpredicted sepal width on average.

Although Estimator 1 was clearly stronger, both approaches remain limited by their use of constant predictions. A useful next step would be to compare them with mean training sepal width as a baseline and then evaluate fitted regression models capable of producing observation-specific predictions.

## Classification Analysis

Species classification was evaluated using two sepal-length threshold rules. Classifier 1 used the 25th and 50th training percentiles, while Classifier 2 used the 50th and 75th percentiles. Performance was assessed using accuracy, precision, recall, and F1-score to examine both overall and class-specific behavior.

In [17]:
from numpy import percentile

# Establish lower classification thresholds from training sepal-length percentiles
est3 = percentile(X_train['sepal length (cm)'], [25, 50])

# Assign species classes according to the percentile thresholds
y_hat = np.zeros(len(y_test))
y_hat[X_test['sepal length (cm)'] > est3[0]] = 1
y_hat[X_test['sepal length (cm)']>est3[1]]=2 
y_hat=y_hat.astype('int') 

# Evaluate overall and class-specific classification performance
print(cr(y_test.astype('int'),y_hat)) 


              precision    recall  f1-score   support

           0       0.71      0.50      0.59        10
           1       0.33      0.20      0.25        10
           2       0.59      1.00      0.74        10

    accuracy                           0.57        30
   macro avg       0.55      0.57      0.53        30
weighted avg       0.55      0.57      0.53        30



In [18]:
# Shift the decision thresholds upward to evaluate an alternative class boundary
est4=percentile(X_train['sepal length (cm)'], [50,75]) 

y_hat2=np.zeros(len(y_test)) 
y_hat2[X_test['sepal length (cm)']>est4[0]]=1 
y_hat2[X_test['sepal length (cm)']>est4[1]]=2 
y_hat2=y_hat2.astype('int') 

# Evaluate the revised threshold rule using the same performance measures
print(cr(y_test.astype('int'),y_hat2)) 

              precision    recall  f1-score   support

           0       0.69      0.90      0.78        10
           1       0.38      0.30      0.33        10
           2       0.67      0.60      0.63        10

    accuracy                           0.60        30
   macro avg       0.58      0.60      0.58        30
weighted avg       0.58      0.60      0.58        30



### Classification Findings

Classifier 2 produced the stronger overall performance, with 60.0% accuracy and a macro F1-score of 0.58 compared with 56.7% accuracy and a macro F1-score of 0.53 for Classifier 1. Classifier 1 achieved perfect recall for Virginica but performed less effectively across the remaining classes. Classifier 2 provided a more balanced result overall, although the improvement represented only one additional correct classification.

Versicolor remained the most difficult class to distinguish, indicating that sepal length alone provides limited separation among all three species.

## Model Limitations and Next Steps

The regression results show that lower error does not eliminate the limitation of constant predictions, while the classification results show that modest gains in overall accuracy can mask substantial differences in class-specific performance.

Future work should establish a simple regression baseline using mean training sepal width, evaluate models that incorporate observation-level predictors, and test additional features for classification. Cross-validation would provide a stronger basis for determining whether apparent performance improvements are stable across different samples. A confusion matrix would also provide additional diagnostic value by showing how misclassifications are distributed across species, particularly for Versicolor, which remained the most difficult class to distinguish.